# Diagnostics (Universal)

This notebook provides a universal diagnostics workflow that works for **any ML type**
(binary classification, multiclass, regression, time-to-event).

**Part 1 — Study Overview** (via `predict/notebook_utils`): study details, performance metrics, selected features.

**Part 2 — Feature Importances** (via `diagnostics`): saved FI parquet files, filterable by method, dataset, etc.

**Part 3 — Optuna Diagnostics** (via `diagnostics`): trial counts, trials scatter, hyperparameters.

No model loading is performed.

---

> **Note:** This notebook uses `ipywidgets` for interactive dropdown selection in the Feature Importance and Optuna sections.
>
> If `ipywidgets` is not installed, the notebook will fall back to static plots with default parameters.
>
> To install: `pip install ipywidgets`

## Imports

In [ ]:
from octopus.predict.notebook_utils import (
    find_latest_study,
    show_study_details,
    show_target_metric_performance,
    show_selected_features,
)
from octopus.diagnostics import StudyDiagnostics

In [ ]:
# Check for ipywidgets availability
try:
    import ipywidgets as widgets
    from IPython.display import display
    from octopus.types import MetricDirection
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("⚠️ ipywidgets not installed. Interactive widgets will not be available.")
    print("Install with: pip install ipywidgets")
    print("Falling back to static plots with default parameters.")

## Input

In [ ]:
study_name_prefix = "wf_octo_mrmr_octo"
studies_root = "../studies"

study_directory = find_latest_study(studies_root, study_name_prefix)
print(f"Using study: {study_directory}")

---
## Part 1: Study Overview

### Study Details

In [ ]:
study_info = show_study_details(study_directory)

### Target Metric Performance

In [ ]:
performance_tables = show_target_metric_performance(study_info)

### Selected Features Summary

In [ ]:
feature_table, freq_table, raw = show_selected_features(study_info)

---
## Part 2: Feature Importances (from saved results)

Two-step workflow:
1. **`get_feature_importances()`** — load raw FI data filtered by task/module (returns a DataFrame)
2. **`plot_feature_importance()`** — plot the FI data with additional filters (fi_method, fi_dataset, etc.)

In [ ]:
diag = StudyDiagnostics(study_info["path"])

### Load raw FI data for task 0

In [ ]:
fi_table = diag.get_feature_importances(task_id=0, result_type="best")
print(f"FI rows: {len(fi_table)}")
for col in ["fi_method", "fi_dataset", "training_id", "module", "outersplit_id"]:
    if col in fi_table.columns:
        print(f"  {col}: {sorted(fi_table[col].unique().tolist())}")
fi_table.head()

### Interactive Feature Importance Explorer

Use the dropdown menus below to explore feature importances by:
- **FI Method**: How importance was computed (`internal`, `permutation`, etc.)
- **FI Dataset**: Which data partition (`train`, `dev`)
- **Outersplit**: Specific outer fold or `all` for aggregated view
- **Top N**: Number of top features to display (5-50)

The chart shows **mean importance** across all training IDs within the selected filters.

In [ ]:
if not HAS_WIDGETS:
    # Fallback: show one example plot
    diag.plot_feature_importance(fi_table, fi_method="internal", fi_dataset="train")
else:
    # Build options from actual data
    if fi_table.empty:
        print("❌ No feature importance data found for this task.")
    else:
        fi_methods = sorted(fi_table["fi_method"].unique().tolist())
        fi_datasets = sorted(fi_table["fi_dataset"].unique().tolist())
        outersplit_ids = sorted(fi_table["outersplit_id"].unique().tolist())
        outersplits = ["all"] + [str(x) for x in outersplit_ids]

        # Create widgets
        w_method = widgets.Dropdown(
            options=fi_methods,
            value=fi_methods[0],
            description="FI Method:",
            style={"description_width": "initial"}
        )
        w_dataset = widgets.Dropdown(
            options=fi_datasets,
            value=fi_datasets[0],
            description="FI Dataset:",
            style={"description_width": "initial"}
        )
        w_outer = widgets.Dropdown(
            options=outersplits,
            value="all",
            description="Outersplit:",
            style={"description_width": "initial"}
        )
        w_topn = widgets.IntSlider(
            value=20,
            min=5,
            max=50,
            step=5,
            description="Top N:",
            style={"description_width": "initial"}
        )

        out = widgets.Output()

        def update_fi_plot(change=None):
            """Update FI plot when any widget changes."""
            out.clear_output(wait=True)
            with out:
                outer = None if w_outer.value == "all" else int(w_outer.value)
                diag.plot_feature_importance(
                    fi_table,
                    fi_method=w_method.value,
                    fi_dataset=w_dataset.value,
                    outersplit_id=outer,
                    top_n=w_topn.value,
                )

        # Attach observers
        w_method.observe(update_fi_plot, names="value")
        w_dataset.observe(update_fi_plot, names="value")
        w_outer.observe(update_fi_plot, names="value")
        w_topn.observe(update_fi_plot, names="value")

        # Display widgets and output
        display(widgets.HBox([w_method, w_dataset, w_outer, w_topn]))
        display(out)

        # Initial render
        update_fi_plot()

### Number of Unique Trials by Model Type

In [ ]:
diag.plot_optuna_trial_counts()

### Interactive Optuna Trials Explorer

Explore optimization progress across different outersplits and tasks:
- **Outersplit**: Which outer CV fold
- **Task**: Which workflow task (if multiple)
- **Direction**: Whether the metric should be minimized or maximized

**Note:** Check your study's target metric to set the correct direction (e.g., RMSE → minimize, AUCROC → maximize).

In [ ]:
if not HAS_WIDGETS:
    # Fallback: show one example plot
    diag.plot_optuna_trials(outersplit_id=0, task_id=0)
else:
    optuna_df = diag.optuna_trials
    if optuna_df.empty:
        print("❌ No Optuna data found.")
    else:
        opt_outersplits = sorted(optuna_df["outersplit_id"].unique().tolist())
        opt_tasks = sorted(optuna_df["task_id"].unique().tolist())

        w_opt_outer = widgets.Dropdown(
            options=opt_outersplits,
            value=opt_outersplits[0],
            description="Outersplit:",
            style={"description_width": "initial"}
        )
        w_opt_task = widgets.Dropdown(
            options=opt_tasks,
            value=opt_tasks[0],
            description="Task:",
            style={"description_width": "initial"}
        )
        w_direction = widgets.Dropdown(
            options=["minimize", "maximize"],
            value="minimize",
            description="Direction:",
            style={"description_width": "initial"}
        )

        out_trials = widgets.Output()

        def update_trials(change=None):
            """Update trials plot when any widget changes."""
            out_trials.clear_output(wait=True)
            with out_trials:
                direction = MetricDirection(w_direction.value)
                diag.plot_optuna_trials(
                    outersplit_id=w_opt_outer.value,
                    task_id=w_opt_task.value,
                    direction=direction,
                )

        w_opt_outer.observe(update_trials, names="value")
        w_opt_task.observe(update_trials, names="value")
        w_direction.observe(update_trials, names="value")

        display(widgets.HBox([w_opt_outer, w_opt_task, w_direction]))
        display(out_trials)
        update_trials()

### Interactive Optuna Hyperparameters Explorer

Visualize hyperparameter distributions and their relationship to objective values:
- **Outersplit**: Which outer CV fold
- **Task**: Which workflow task
- **Model Type**: Filter by specific model (e.g., `xgboost`, `lightgbm`) or leave empty for all models

In [ ]:
if not HAS_WIDGETS:
    # Fallback: show one example plot
    diag.plot_optuna_hyperparameters(outersplit_id=0, task_id=0)
else:
    if optuna_df.empty:
        print("❌ No Optuna data found.")
    else:
        # Reuse outersplit/task options or create new ones
        opt_outersplits = sorted(optuna_df["outersplit_id"].unique().tolist())
        opt_tasks = sorted(optuna_df["task_id"].unique().tolist())
        model_types = [""] + sorted(optuna_df["model_type"].unique().tolist())

        w_hp_outer = widgets.Dropdown(
            options=opt_outersplits,
            value=opt_outersplits[0],
            description="Outersplit:",
            style={"description_width": "initial"}
        )
        w_hp_task = widgets.Dropdown(
            options=opt_tasks,
            value=opt_tasks[0],
            description="Task:",
            style={"description_width": "initial"}
        )
        w_model = widgets.Dropdown(
            options=model_types,
            value="",
            description="Model Type:",
            style={"description_width": "initial"}
        )

        out_hp = widgets.Output()

        def update_hp(change=None):
            """Update hyperparameters plot when any widget changes."""
            out_hp.clear_output(wait=True)
            with out_hp:
                model = w_model.value if w_model.value else ""
                diag.plot_optuna_hyperparameters(
                    outersplit_id=w_hp_outer.value,
                    task_id=w_hp_task.value,
                    model_type=model,
                )

        w_hp_outer.observe(update_hp, names="value")
        w_hp_task.observe(update_hp, names="value")
        w_model.observe(update_hp, names="value")

        display(widgets.HBox([w_hp_outer, w_hp_task, w_model]))
        display(out_hp)
        update_hp()